# Distributed Training at Scale

Modern deep learning models GPT-4, LLaMA, Gemini have billions to trillions of parameters. Training them on a single GPU is impossible. This notebook covers every major technique used to distribute training across dozens, hundreds, or thousands of GPUs.

## What You Will Learn

| Topic | Key Idea |
|---|---|
| Data Parallelism (DDP) | Replicate model, split data, average gradients |
| Model Parallelism | Split layers across GPUs |
| Pipeline Parallelism | Split layers into stages, use micro-batches |
| Tensor Parallelism | Split individual matrices across GPUs |
| ZeRO / FSDP | Shard optimizer state, gradients, and parameters |
| DeepSpeed | ZeRO + CPU/NVMe offload |
| Mixed Precision | FP16/BF16 + gradient scaling |
| Gradient Checkpointing | Trade compute for memory |
| FlashAttention | IO-aware attention, $O(N)$ memory |

**Prerequisites:** Basic PyTorch, neural network fundamentals, familiarity with GPUs.

## 1. Why Distributed Training?

### The Memory Wall

A single A100 (80 GB) cannot hold GPT-3 (175B params) even in float16:

$$\text{Parameter memory (FP16)} = 175 \times 10^9 \times 2 \text{ bytes} \approx 350 \text{ GB}$$

With optimizer states (Adam stores 2 moments in FP32), gradients, and activations:

$$\text{Total} \approx 16\Psi \text{ bytes} \quad (\Psi = \text{number of parameters})$$

For GPT-3: $16 \times 175 \times 10^9 \approx 2.8 \text{ TB}$. You need ~35 A100s just to fit the model.

### The Compute Wall

Even if memory were unlimited, training GPT-3 on one GPU (312 TFLOP/s A100) at 50% MFU would take:

$$t \approx \frac{3.14 \times 10^{23} \text{ FLOPs}}{312 \times 10^{12} \text{ FLOP/s} \times 0.5} \approx 636 \text{ years}$$

With 1024 A100s at 50% MFU: ~226 days. With 10,000 A100s: ~23 days.

### The Solution Space

```
Distributed Training
├── Parallelism strategies
│   ├── Data Parallelism (DP / DDP)
│   ├── Model Parallelism (MP)
│   ├── Pipeline Parallelism (PP)
│   ├── Tensor Parallelism (TP)
│   ├── Sequence Parallelism (SP)
│   └── Context Parallelism (CP)
├── Memory optimizations
│   ├── ZeRO-1/2/3
│   ├── FSDP
│   ├── Gradient checkpointing
│   └── Activation offloading
└── Compute optimizations
    ├── Mixed precision (FP16, BF16)
    └── FlashAttention
```

## 2. Data Parallelism and DDP

### Core Idea

Each GPU holds a **full copy** of the model. The global batch is split into $N$ mini-batches (one per GPU). After the forward and backward pass, gradients are **averaged** across GPUs:

$$g_i^{\text{sync}} = \frac{1}{N} \sum_{k=1}^{N} g_i^{(k)}$$

All replicas then take the same optimizer step, staying in sync.

### All-Reduce

The gradient averaging is implemented with an **All-Reduce** collective. The ring-allreduce algorithm (used by NCCL) achieves:

$$\text{Communication volume per GPU} = 2 \cdot \frac{N-1}{N} \cdot \Psi$$

For large $N$: $\approx 2\Psi$ independent of the number of GPUs! This is why DDP scales well.

### DDP vs DP

| Feature | `nn.DataParallel` (DP) | `nn.parallel.DistributedDataParallel` (DDP) |
|---|---|---|
| Architecture | Single-process, multi-thread | Multi-process, one per GPU |
| GIL bottleneck | Yes | No |
| Gradient sync | Parameter server (rank 0 bottleneck) | Ring all-reduce |
| Overlap comm/compute | No | Yes (gradient bucketing) |
| Recommended | No | Yes |

### Gradient Bucketing

DDP does not all-reduce each gradient individually. It groups them into **buckets** (default 25 MB) and overlaps the all-reduce of completed buckets with the backward pass of remaining layers hiding most communication latency.

In [1]:
# ============================================================
# Data Parallelism with torch.distributed (DDP)
# ============================================================
# In production you would launch with:
#   torchrun --nproc_per_node=4 script.py
# Here we show the structure without actually launching processes.

import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler


def build_model(hidden: int = 512) -> nn.Module:
    return nn.Sequential(
        nn.Linear(784, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 10),
    )


def ddp_train(rank: int, world_size: int):
    """Entry point for each worker process."""
    # 1. Initialize the process group
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group(backend="nccl", rank=rank, world_size=world_size)

    # 2. Set device
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    # 3. Build model and wrap with DDP
    model = build_model().to(device)
    model = DDP(model, device_ids=[rank], find_unused_parameters=False)
    #           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    #           DDP registers hooks on each parameter to trigger
    #           all-reduce as soon as its gradient is ready.

    # 4. Distributed sampler ensures each rank sees different data
    dataset = TensorDataset(
        torch.randn(1000, 784),
        torch.randint(0, 10, (1000,)),
    )
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, batch_size=32, sampler=sampler)

    # 5. Standard training loop
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(3):
        sampler.set_epoch(epoch)  # Ensures different shuffling each epoch
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()  # <-- gradients all-reduced here automatically
            optimizer.step()

    # 6. Save only from rank 0
    if rank == 0:
        torch.save(model.module.state_dict(), "checkpoint.pt")
        print("Saved checkpoint from rank 0")

    dist.destroy_process_group()


# To run: torch.multiprocessing.spawn(ddp_train, args=(4,), nprocs=4)
print("DDP training function defined.")
print(f"NCCL available: {torch.cuda.nccl.is_available([]) if torch.cuda.is_available() else 'N/A (no CUDA)'}")

DDP training function defined.
NCCL available: N/A (no CUDA)


## 3. Model Parallelism

When a model is too large to fit on a single GPU, we partition its **layers** across GPUs. This is called **model parallelism** (also called **tensor parallelism** when individual tensors are split, or **pipeline parallelism** when layers are grouped into stages).

### Naive Model Parallelism

Split consecutive layer groups across GPUs:

```
GPU 0: Embedding + Layers 0-11
GPU 1: Layers 12-23
GPU 2: Layers 24-35
GPU 3: Layers 36-47 + LM Head
```

**Problem:** At any moment, only one GPU is active. GPU utilization is ~$1/N$.

### The Bubble Problem

With a single micro-batch, the pipeline bubble fraction is:

$$\text{Bubble fraction} = \frac{p-1}{m + p - 1}$$

where $p$ = pipeline stages, $m$ = number of micro-batches.

For $p=4, m=1$: bubble = $3/4 = 75\%$. For $p=4, m=8$: bubble = $3/11 \approx 27\%$.

**Solution:** Use many micro-batches to keep all stages busy (see Pipeline Parallelism).

In [2]:
# ============================================================
# Naive Model Parallelism Example (CPU-based demo)
# ============================================================
import torch
import torch.nn as nn


class NaiveModelParallelMLP(nn.Module):
    """Splits a 4-layer MLP across 2 devices.
    In real use, replace 'cpu' with 'cuda:0' / 'cuda:1'.
    """

    def __init__(self, device0="cpu", device1="cpu"):
        super().__init__()
        self.device0 = device0
        self.device1 = device1

        # First half on device0
        self.part1 = nn.Sequential(
            nn.Linear(784, 2048),
            nn.ReLU(),
            nn.Linear(2048, 2048),
            nn.ReLU(),
        ).to(device0)

        # Second half on device1
        self.part2 = nn.Sequential(
            nn.Linear(2048, 2048),
            nn.ReLU(),
            nn.Linear(2048, 10),
        ).to(device1)

    def forward(self, x):
        # Move intermediate activations between devices
        x = self.part1(x.to(self.device0))
        x = self.part2(x.to(self.device1))  # D2H + H2D transfer
        return x


model = NaiveModelParallelMLP()
x = torch.randn(16, 784)
out = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

# Count params per partition
params_p1 = sum(p.numel() for p in model.part1.parameters())
params_p2 = sum(p.numel() for p in model.part2.parameters())
print(f"Params on device0: {params_p1:,}")
print(f"Params on device1: {params_p2:,}")

Input shape:  torch.Size([16, 784])
Output shape: torch.Size([16, 10])
Params on device0: 5,804,032
Params on device1: 4,216,842


## 4. Pipeline Parallelism

Pipeline parallelism splits layers into **stages** and processes multiple **micro-batches** simultaneously to keep all stages busy.

### GPipe (Google, 2019)

- Splits mini-batch into $m$ micro-batches.
- Forward pass: feed micro-batches one by one down the pipeline.
- Backward pass: after all micro-batches complete the forward pass, do backprop in reverse order.
- **Gradient accumulation** happens across micro-batches before the optimizer step.
- **Memory cost:** Must store all $m$ sets of activations for backprop $O(m \times \text{activation\_size})$.
- Bubble fraction: $\frac{p-1}{m + p - 1}$

### PipeDream (Microsoft, 2019)

- **1F1B schedule:** Each stage alternates between one forward and one backward micro-batch.
- Significantly reduces the pipeline bubble.
- Requires storing **multiple versions** of weights (weight stashing) weights used in forward must be available during the corresponding backward.
- Memory: $O(p)$ versions of weights instead of $O(m)$ activations.

### 1F1B Schedule Visualization

```
Time:   1   2   3   4   5   6   7   8
GPU0:  F1  F2  F3  F4  B4  B3  B2  B1
GPU1:      F1  F2  F3  F4  B4  B3  B2  B1
GPU2:          F1  F2  F3  F4  B4  B3  B2  B1
GPU3:              F1  F2  F3  F4  B4  B3  B2  B1
```

Steady state: all GPUs busy. Bubble only at startup/teardown.

### Interleaved Stages (Megatron-LM v2)

Each GPU holds $v$ non-consecutive model chunks (virtual stages). This reduces the bubble fraction to:

$$\text{Bubble fraction} = \frac{1}{v} \cdot \frac{p-1}{m + p - 1}$$

At the cost of $v$ times more point-to-point communication.

## 5. Tensor Parallelism (Megatron-LM Style)

Tensor parallelism splits **individual weight matrices** across GPUs. This is the approach used in Megatron-LM.

### Column-then-Row Splitting for MLP

Consider the two-layer MLP: $Y = \text{GeLU}(XA)$, $Z = YB$.

**Column parallel (first layer):** Split $A$ by columns: $A = [A_1, A_2]$

$$XA = [XA_1, XA_2]$$

Each GPU $i$ computes $Y_i = \text{GeLU}(XA_i)$ no communication needed for GeLU since it is element-wise.

**Row parallel (second layer):** Split $B$ by rows: $B = \begin{bmatrix} B_1 \\ B_2 \end{bmatrix}$

$$Z = YB = Y_1 B_1 + Y_2 B_2$$

Each GPU computes its partial sum, then an **All-Reduce** gives the final $Z$.

**Communication cost:** 2 All-Reduces per transformer layer (one in MLP, one in attention).

### Transformer Attention Splitting

Multi-head attention: split heads across GPUs. Each GPU handles $H/N$ heads:

$$\text{Attention}(Q, K, V) \text{ with } H \text{ heads} \rightarrow \text{each GPU handles } H/N \text{ heads}$$

### 3D Parallelism (Megatron + Pipeline + Data)

Megatron-LM combines:
- **Tensor Parallelism** (TP): within a node (NVLink bandwidth)
- **Pipeline Parallelism** (PP): across nodes, fewer stages
- **Data Parallelism** (DP): replicate the TP+PP model

Total GPUs: $N_{\text{total}} = N_{TP} \times N_{PP} \times N_{DP}$

For LLaMA-3 70B on 128 A100s: TP=8, PP=4, DP=4.

## 6. Sequence Parallelism and Context Parallelism

### Sequence Parallelism (Megatron-LM v3)

In tensor parallelism, operations outside the column/row split (LayerNorm, Dropout) are replicated on every GPU wasting memory. Sequence parallelism splits the **sequence dimension** across TP ranks for these operations.

```
LayerNorm     [S/tp, B, H] , sequence split
   |  all-gather
Attention     [S, B, H]    , full sequence for QKV
   |  reduce-scatter
LayerNorm     [S/tp, B, H] , sequence split
```

This replaces the all-reduce with all-gather + reduce-scatter same communication volume but enables activation memory reduction by $N_{TP}$.

### Context Parallelism (CP)

For very long sequences (e.g., 128K+ tokens), even a single attention layer's $O(N^2)$ attention matrix is too large. Context parallelism splits the sequence across CP ranks:

- Each rank holds $S/N_{CP}$ tokens' Q, K, V.
- For full attention, every rank needs all K, V use ring-style all-gather.
- Implemented in Megatron-LM's "ring attention" variant.

**Memory saving:** Attention activations reduced by $N_{CP}$.

### Ring Attention

```
GPU 0 holds Q[0:S/4], K[0:S/4], V[0:S/4]
GPU 1 holds Q[S/4:S/2], K[S/4:S/2], V[S/4:S/2]
...

Step 1: Each GPU computes attention of its Q against its local KV
Step 2: Send KV to next GPU in ring, receive from previous
Step 3: Compute attention of local Q against received KV, accumulate
... repeat until all KV pairs processed
```

## 7. ZeRO: Zero Redundancy Optimizer

ZeRO (Rajbhandari et al., Microsoft 2020) eliminates the memory redundancy in data parallelism by sharding optimizer states, gradients, and parameters across DP ranks.

### Memory Breakdown (Mixed Precision Training)

For a model with $\Psi$ parameters:

| Component | Bytes per param | Notes |
|---|---|---|
| FP16 parameters | $2\Psi$ | Forward/backward |
| FP16 gradients | $2\Psi$ | Backward pass |
| FP32 master weights | $4\Psi$ | Optimizer copy |
| FP32 Adam momentum | $4\Psi$ | 1st moment |
| FP32 Adam variance | $4\Psi$ | 2nd moment |
| **Total** | $16\Psi$ | |

### ZeRO Stages

**ZeRO Stage 1 Partition optimizer states:**

$$M_{\text{ZeRO-1}} = 2\Psi + 2\Psi + \frac{4\Psi + 4\Psi + 4\Psi}{N_{dp}} = 4\Psi + \frac{12\Psi}{N_{dp}}$$

**ZeRO Stage 2 Partition optimizer states + gradients:**

$$M_{\text{ZeRO-2}} = 2\Psi + \frac{2\Psi + 12\Psi}{N_{dp}} = 2\Psi + \frac{14\Psi}{N_{dp}}$$

**ZeRO Stage 3 Partition everything (parameters too):**

$$\boxed{M_{\text{ZeRO-3}} = \frac{16\Psi}{N_{dp}} \text{ bytes}}$$

For $N_{dp} = 64$ GPUs and GPT-3 ($\Psi = 175 \times 10^9$):

$$M_{\text{ZeRO-3}} = \frac{16 \times 175 \times 10^9}{64} \approx 43.75 \text{ GB per GPU}$$

This fits on an A100 80GB!

### ZeRO Communication Pattern

ZeRO-3 requires:
- **Forward pass:** All-gather parameters layer by layer, discard after use.
- **Backward pass:** All-gather parameters for recomputation, reduce-scatter gradients.
- **Optimizer step:** Each rank updates its shard of parameters locally.

Total communication: $3\Psi$ elements (vs $2\Psi$ for DDP all-reduce). The 1.5x overhead is the cost of full parameter sharding.

## 8. FSDP: Fully Sharded Data Parallel

**FSDP** is PyTorch's native implementation of ZeRO-3, introduced in PyTorch 1.12. It integrates deeply with `nn.Module` for easy use.

### How FSDP Works

```
Before forward:
  all_gather(weight_shard) -> full weight  [on all ranks]

Forward:
  compute with full weight
  
After forward:
  free full weight, keep only shard       [free ~(N-1)/N memory]

Backward:
  all_gather(weight_shard) -> full weight  [needed for grad computation]
  compute gradients
  reduce_scatter(gradients) -> grad_shard  [each rank keeps 1/N of grads]
  free full weight and full gradients

Optimizer step:
  update local weight_shard using grad_shard
```

### FSDP Sharding Strategies

| Strategy | Parameters | Gradients | Optimizer State |
|---|---|---|---|
| `NO_SHARD` | Full (= DDP) | Full | Full |
| `SHARD_GRAD_OP` | Full during fwd/bwd | Sharded | Sharded |
| `FULL_SHARD` (ZeRO-3) | Sharded | Sharded | Sharded |
| `HYBRID_SHARD` | Sharded within node | Replicated across nodes | Sharded within node |

### Mixed Precision with FSDP

FSDP's `MixedPrecision` policy allows different dtypes for:
- **param_dtype:** dtype for forward computation (BF16/FP16)
- **reduce_dtype:** dtype for gradient reduction (FP32 for numerical stability)
- **buffer_dtype:** dtype for non-parameter buffers

In [3]:
# ============================================================
# FSDP Example
# ============================================================
import torch
import torch.nn as nn

# These imports require PyTorch >= 1.12
try:
    from torch.distributed.fsdp import (
        FullyShardedDataParallel as FSDP,
        ShardingStrategy,
        MixedPrecision,
        BackwardPrefetch,
        CPUOffload,
    )
    from torch.distributed.fsdp.wrap import (
        size_based_auto_wrap_policy,
        transformer_auto_wrap_policy,
    )
    import functools
    FSDP_AVAILABLE = True
except ImportError:
    FSDP_AVAILABLE = False
    print("FSDP requires PyTorch >= 1.12")


class TransformerBlock(nn.Module):
    """Minimal transformer block to demonstrate FSDP wrapping."""
    def __init__(self, d_model: int = 512, nhead: int = 8):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ff(x))
        return x


class SmallTransformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, n_layers=4, vocab_size=50257):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, nhead) for _ in range(n_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(x)


if FSDP_AVAILABLE:
    # BF16 mixed precision policy (preferred over FP16 on Ampere+)
    bf16_policy = MixedPrecision(
        param_dtype=torch.bfloat16,
        reduce_dtype=torch.bfloat16,
        buffer_dtype=torch.bfloat16,
    )

    # Auto-wrap policy: wrap TransformerBlock modules individually
    # This is the key to FSDP efficiency each block is sharded separately
    wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerBlock},
    )

    # Example FSDP configuration (would be called inside a distributed process)
    fsdp_config = dict(
        sharding_strategy=ShardingStrategy.FULL_SHARD,       # ZeRO-3
        mixed_precision=bf16_policy,
        auto_wrap_policy=wrap_policy,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,     # Overlap comm with compute
        cpu_offload=CPUOffload(offload_params=False),         # Set True for CPU offload
        device_id=torch.cuda.current_device() if torch.cuda.is_available() else None,
        sync_module_states=True,   # Broadcast init weights from rank 0
        limit_all_gathers=True,    # Reduce peak memory by limiting prefetching
    )

    print("FSDP configuration:")
    for k, v in fsdp_config.items():
        print(f"  {k}: {v}")

    # model = FSDP(SmallTransformer(), **fsdp_config)
    # optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    # (rest of training loop same as DDP)
    print("\nTo wrap model: model = FSDP(SmallTransformer(), **fsdp_config)")
else:
    print("Showing FSDP config structure (FSDP import unavailable):")
    print("""
    FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        mixed_precision=MixedPrecision(param_dtype=torch.bfloat16, ...),
        auto_wrap_policy=transformer_auto_wrap_policy(...),
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
    )
    """)

# Verify model structure
model = SmallTransformer()
total_params = sum(p.numel() for p in model.parameters())
print(f"\nSmallTransformer total parameters: {total_params:,}")
print(f"Memory (FP32): {total_params * 4 / 1e6:.1f} MB")
print(f"Memory (BF16): {total_params * 2 / 1e6:.1f} MB")

FSDP configuration:
  sharding_strategy: ShardingStrategy.FULL_SHARD
  mixed_precision: MixedPrecision(param_dtype=torch.bfloat16, reduce_dtype=torch.bfloat16, buffer_dtype=torch.bfloat16, keep_low_precision_grads=False, cast_forward_inputs=False, cast_root_forward_inputs=True, _module_classes_to_ignore=(<class 'torch.nn.modules.batchnorm._BatchNorm'>,))
  auto_wrap_policy: functools.partial(<function transformer_auto_wrap_policy at 0x78cf4e69b740>, transformer_layer_cls={<class '__main__.TransformerBlock'>})
  backward_prefetch: BackwardPrefetch.BACKWARD_PRE
  cpu_offload: CPUOffload(offload_params=False)
  device_id: None
  sync_module_states: True
  limit_all_gathers: True

To wrap model: model = FSDP(SmallTransformer(), **fsdp_config)



SmallTransformer total parameters: 64,072,704
Memory (FP32): 256.3 MB
Memory (BF16): 128.1 MB


## 9. DeepSpeed

DeepSpeed (Microsoft) is an optimization library that implements ZeRO plus additional techniques: CPU offload, NVMe offload, sparse attention, and 1-bit Adam.

### DeepSpeed ZeRO Stages

| Stage | What is sharded | Max model size (8 GPUs, 80GB each) |
|---|---|---|
| ZeRO-1 | Optimizer states | ~3x vs DDP |
| ZeRO-2 | + Gradients | ~8x vs DDP |
| ZeRO-3 | + Parameters | ~64x vs DDP |
| ZeRO-Infinity | + CPU + NVMe offload | Limited by CPU RAM / NVMe |

### CPU Offload

With `cpu_offload`, optimizer states (and optionally parameters) are stored in CPU RAM during the optimizer step. This frees GPU memory but adds PCIe transfer overhead. Effective when GPU memory is the bottleneck.

### NVMe Offload (ZeRO-Infinity)

Parameters and optimizer states can be offloaded to NVMe SSDs. With modern NVMe drives (7 GB/s), a single DGX A100 node can effectively train models with **tens of trillions** of parameters.

### Activation Checkpointing in DeepSpeed

DeepSpeed integrates with gradient checkpointing and also supports **activation partitioning** across DP ranks to reduce per-GPU activation memory by $N_{dp}$.

In [4]:
# ============================================================
# DeepSpeed Configuration JSONs
# ============================================================
import json

# --- ZeRO Stage 2 config (good default for most fine-tuning) ---
zero2_config = {
    "zero_optimization": {
        "stage": 2,
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True
    },
    "bf16": {"enabled": True},
    "train_micro_batch_size_per_gpu": 4,
    "gradient_accumulation_steps": 4,
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": 2e-5,
            "betas": [0.9, 0.999],
            "eps": 1e-8,
            "weight_decay": 0.01
        }
    },
    "scheduler": {
        "type": "WarmupDecayLR",
        "params": {
            "warmup_min_lr": 0,
            "warmup_max_lr": 2e-5,
            "warmup_num_steps": 500,
            "total_num_steps": 10000
        }
    }
}

# --- ZeRO Stage 3 with CPU offload ---
zero3_cpu_config = {
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "offload_param": {
            "device": "cpu",
            "pin_memory": True
        },
        "overlap_comm": True,
        "contiguous_gradients": True,
        "sub_group_size": 1e9,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "bf16": {"enabled": True},
    "gradient_checkpointing": True,
    "train_micro_batch_size_per_gpu": 1,
    "gradient_accumulation_steps": 16,
}

# --- ZeRO-Infinity (NVMe offload) ---
zero_infinity_config = {
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "nvme",
            "nvme_path": "/local_nvme",
            "pin_memory": True,
            "buffer_count": 4,
            "fast_init": False
        },
        "offload_param": {
            "device": "nvme",
            "nvme_path": "/local_nvme",
            "pin_memory": True,
            "buffer_count": 5,
            "buffer_size": 1e8,
            "max_in_cpu": 1e9
        },
        "aio": {
            "block_size": 262144,
            "queue_depth": 32,
            "thread_count": 1,
            "single_submit": False,
            "overlap_events": True
        }
    }
}

print("=== ZeRO Stage 2 Config ===")
print(json.dumps(zero2_config["zero_optimization"], indent=2))

print("\n=== ZeRO Stage 3 + CPU Offload (key fields) ===")
print(json.dumps({k: v for k, v in zero3_cpu_config["zero_optimization"].items()
                  if k in ["stage", "offload_optimizer", "offload_param"]}, indent=2))

# Usage:
# import deepspeed
# model_engine, optimizer, _, _ = deepspeed.initialize(
#     model=model,
#     config=zero2_config,
#     model_parameters=model.parameters()
# )

=== ZeRO Stage 2 Config ===
{
  "stage": 2,
  "allgather_partitions": true,
  "allgather_bucket_size": 200000000.0,
  "overlap_comm": true,
  "reduce_scatter": true,
  "reduce_bucket_size": 200000000.0,
  "contiguous_gradients": true
}

=== ZeRO Stage 3 + CPU Offload (key fields) ===
{
  "stage": 3,
  "offload_optimizer": {
    "device": "cpu",
    "pin_memory": true
  },
  "offload_param": {
    "device": "cpu",
    "pin_memory": true
  }
}


## 10. Horovod and Communication Backends

### Horovod (Uber, 2018)

Horovod is a distributed training framework that wraps existing training code with minimal changes. It uses the **ring-allreduce** algorithm (from Baidu's "Bringing HPC Techniques to Deep Learning").

**Key features:**
- Framework-agnostic: TensorFlow, PyTorch, MXNet, Keras
- Tight MPI integration
- Tensor fusion: batches small tensors before all-reduce
- Timeline profiling tool

```python
import horovod.torch as hvd

hvd.init()
torch.cuda.set_device(hvd.local_rank())

optimizer = torch.optim.SGD(model.parameters(), lr=0.01 * hvd.size())
optimizer = hvd.DistributedOptimizer(optimizer, named_parameters=model.named_parameters())
hvd.broadcast_parameters(model.state_dict(), root_rank=0)
```

### Communication Backends

| Backend | Transport | Best For |
|---|---|---|
| **NCCL** | NVLink, InfiniBand, RoCE | GPU-GPU comm (default for multi-GPU) |
| **Gloo** | Ethernet, shared memory | CPU tensors, debugging, macOS |
| **MPI** | Any MPI transport | HPC clusters with MPI support |

### NCCL Topology Awareness

NCCL automatically detects the hardware topology (NVLink vs PCIe vs InfiniBand) and selects the optimal all-reduce algorithm:

- **NVLink within node:** Ring or tree all-reduce at ~600 GB/s (A100 NVLink)
- **InfiniBand across nodes:** RDMA-based all-reduce at 200 Gb/s HDR
- **PCIe only:** Slower, bottlenecked by PCIe bandwidth (~32 GB/s per slot)

### Initializing the Process Group

```python
import torch.distributed as dist

# For GPU training:
dist.init_process_group(backend="nccl")

# For CPU-only:
dist.init_process_group(backend="gloo")

# Custom init:
dist.init_process_group(
    backend="nccl",
    init_method="tcp://master_ip:23456",
    world_size=world_size,
    rank=rank,
    timeout=datetime.timedelta(minutes=30),
)
```

## 11. Mixed Precision Training

### FP32 vs FP16 vs BF16

| Format | Exponent bits | Mantissa bits | Max value | Dynamic range |
|---|---|---|---|---|
| FP32 | 8 | 23 | ~3.4e38 | ~1.7e38 |
| FP16 | 5 | 10 | 65504 | ~65504 |
| BF16 | 8 | 7 | ~3.4e38 | ~1.7e38 |

**FP16 advantages:** 2x memory vs FP32, 2x faster on tensor cores.
**FP16 problem:** Small dynamic range causes overflow (loss = NaN) and underflow (gradient = 0).
**BF16 solution:** Same exponent range as FP32 (no overflow), but less precision. Preferred for LLM training on Ampere/Hopper GPUs.

### Gradient Scaling (for FP16)

To prevent FP16 gradient underflow, scale the loss by a large factor $S$ before backward, then unscale before the optimizer step:

$$\mathcal{L}_{\text{scaled}} = S \cdot \mathcal{L}$$

$$g_{\text{scaled}} = S \cdot g \quad \text{(stored in FP16, no underflow)}$$

$$g_{\text{true}} = \frac{g_{\text{scaled}}}{S} \quad \text{(unscaled before optimizer step)}$$

The scalar $S$ is dynamically adjusted: increased if no inf/nan detected, decreased if overflow occurs.

### AMP (Automatic Mixed Precision)

PyTorch's `torch.cuda.amp` automates this:
- `autocast`: Automatically casts ops to FP16/BF16 where safe (matmul, conv) and keeps FP32 where needed (softmax, LayerNorm).
- `GradScaler`: Handles loss scaling automatically.

In [5]:
# ============================================================
# Mixed Precision Training with torch.cuda.amp
# ============================================================
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler


def train_with_amp(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loader,
    device: torch.device,
    use_bf16: bool = True,
    n_epochs: int = 3,
):
    """
    Training loop with Automatic Mixed Precision.
    
    BF16 is preferred on Ampere+ (A100, H100) no scaling needed.
    FP16 is used on Volta/Turing (V100, T4) requires GradScaler.
    """
    dtype = torch.bfloat16 if use_bf16 else torch.float16
    
    # GradScaler is only needed for FP16 (BF16 has FP32 dynamic range)
    scaler = GradScaler(enabled=(not use_bf16 and device.type == "cuda"))
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    for epoch in range(n_epochs):
        total_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)  # set_to_none=True saves memory

            # autocast context: eligible ops run in lower precision
            with autocast(device_type=device.type, dtype=dtype):
                logits = model(x)
                loss = criterion(logits, y)

            if scaler.is_enabled():
                # FP16 path: scale loss, backward, unscale, step
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)   # Unscale before gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)       # Skips step if inf/nan gradients
                scaler.update()              # Adjusts scale factor
            else:
                # BF16 path: no scaling needed
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}: avg loss = {total_loss / len(loader):.4f}, "
              f"scale = {scaler.get_scale():.0f}")


# Demo with CPU (no actual mixed precision, but shows API)
model = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 10))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(torch.randn(200, 64), torch.randint(0, 10, (200,)))
loader = DataLoader(dataset, batch_size=32)

print(f"Training on: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")
    train_with_amp(model, optimizer, loader, device, use_bf16=torch.cuda.is_bf16_supported())
else:
    # CPU training no AMP benefit, but shows the structure
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        for x, y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
    print("CPU training complete (AMP has no effect on CPU).")

print("\nMemory comparison (FP32 vs BF16 for 175B param model):")
params = 175e9
print(f"  FP32:  {params * 4 / 1e12:.1f} TB")
print(f"  BF16:  {params * 2 / 1e12:.1f} TB")
print(f"  Speedup from tensor cores: ~2x for GEMM operations")

Training on: cpu
CUDA available: False
CPU training complete (AMP has no effect on CPU).

Memory comparison (FP32 vs BF16 for 175B param model):
  FP32:  0.7 TB
  BF16:  0.3 TB
  Speedup from tensor cores: ~2x for GEMM operations


## 12. Gradient Checkpointing

### The Memory-Compute Trade-off

During the backward pass, PyTorch needs activations from the forward pass to compute gradients. By default, **all** intermediate activations are kept in memory.

For a transformer with $L$ layers, each layer with activation size $A$:
$$M_{\text{activations}} = L \times A$$

For GPT-3 with sequence length 2048, this is ~5 TB just for activations.

### Gradient Checkpointing (Activation Recomputation)

Instead of storing all activations, **discard** them after the forward pass and **recompute** them during backprop when needed.

**Full recomputation:**
- Memory: $O(1)$ activations (only need current layer's input)
- Compute cost: +33% (one extra forward pass per backward pass)

**Selective checkpointing (Chen et al., 2016):**
- Checkpoint every $\sqrt{L}$ layers
- Memory: $O(\sqrt{L})$ square root reduction
- Compute cost: one extra forward pass

$$M_{\text{with\_checkpointing}} = O\!\left(\sqrt{L} \cdot A\right) \quad \text{vs} \quad O(L \cdot A)$$

### What to Checkpoint

In practice, transformers checkpoint at the **transformer block** level:
- Store: block input activations
- Discard: QKV matrices, attention weights, FFN intermediate activations
- Recompute during backward: the entire block's forward pass

This saves ~40-50% activation memory for typical transformer architectures.

In [6]:
# ============================================================
# Gradient Checkpointing Example
# ============================================================
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint, checkpoint_sequential


class CheckpointedTransformerBlock(nn.Module):
    """Transformer block that uses gradient checkpointing."""

    def __init__(self, d_model: int = 256, nhead: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.use_checkpoint = True

    def _forward_impl(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ff(x))
        return x

    def forward(self, x):
        if self.use_checkpoint and x.requires_grad:
            # checkpoint() recomputes this function during backward
            # use_reentrant=False is the modern, preferred API
            return checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)


def measure_memory_usage(model, x, label):
    """Run a forward+backward pass and report peak memory."""
    if not torch.cuda.is_available():
        # CPU demo: just show the forward pass works
        y = model(x)
        loss = y.sum()
        loss.backward()
        print(f"{label}: forward+backward OK (CUDA needed for memory measurement)")
        return

    torch.cuda.reset_peak_memory_stats()
    y = model(x)
    loss = y.sum()
    loss.backward()
    peak = torch.cuda.max_memory_allocated() / 1e6
    print(f"{label}: peak memory = {peak:.1f} MB")


d_model, n_layers, seq_len, batch = 256, 8, 512, 4

# Model WITHOUT checkpointing
layers_no_ckpt = nn.ModuleList([
    CheckpointedTransformerBlock(d_model) for _ in range(n_layers)
])
for layer in layers_no_ckpt:
    layer.use_checkpoint = False

model_no_ckpt = nn.Sequential(*layers_no_ckpt)

# Model WITH checkpointing
model_ckpt = nn.Sequential(*[
    CheckpointedTransformerBlock(d_model) for _ in range(n_layers)
])

# Input requires grad so checkpointing is triggered
x = torch.randn(batch, seq_len, d_model, requires_grad=True)

# Without checkpointing
def forward_no_ckpt(x):
    for layer in model_no_ckpt:
        x = layer(x)
    return x

# With checkpointing (per-layer)
def forward_ckpt(x):
    for layer in model_ckpt:
        x = layer(x)
    return x

print("Running gradient checkpointing demo...")
measure_memory_usage(model_no_ckpt, x.detach().requires_grad_(True), "Without checkpointing")
measure_memory_usage(model_ckpt, x.detach().requires_grad_(True), "With checkpointing   ")

print("\ncheckpoint_sequential: checkpoints every k segments")
# checkpoint_sequential(layers, segments, input)
# Useful for sequential models checkpoint every len(layers)//4 layers
print("  checkpoint_sequential(model, segments=4, input=x)")
print("  -> Checkpoints 4 evenly-spaced points in the sequence")
print(f"  -> Saves ~{(1 - 1/4)*100:.0f}% of activation memory vs no checkpointing")

Running gradient checkpointing demo...


Without checkpointing: forward+backward OK (CUDA needed for memory measurement)


With checkpointing   : forward+backward OK (CUDA needed for memory measurement)

checkpoint_sequential: checkpoints every k segments
  checkpoint_sequential(model, segments=4, input=x)
  -> Checkpoints 4 evenly-spaced points in the sequence
  -> Saves ~75% of activation memory vs no checkpointing


## 13. Activation Offloading

Gradient checkpointing recomputes activations at the cost of extra compute. An alternative is **activation offloading**: store activations in CPU memory during the forward pass and bring them back to GPU during the backward pass.

### Trade-off Analysis

| Strategy | GPU Memory | Compute Overhead | PCIe Bandwidth Required |
|---|---|---|---|
| No optimization | $O(L \cdot A)$ | 1x | 0 |
| Gradient checkpointing | $O(A)$ | ~1.33x | 0 |
| Activation offloading | $O(A)$ | ~1x (if hidden by compute) | High |
| Both | $O(A/L)$ | ~1.33x | Moderate |

### When to Use Activation Offloading

- GPU memory is the bottleneck but PCIe bandwidth is available.
- GPU compute is slow enough that CPU transfers can be hidden.
- Long sequences (e.g., 128K tokens) where activations dominate memory.

### Implementation Sketch

```python
class OffloadedActivationHook:
    """Store activations on CPU, retrieve during backward."""
    
    def __init__(self):
        self.cpu_storage = {}
    
    def forward_hook(self, module, input, output):
        # Move activations to CPU after each layer
        self.cpu_storage[id(module)] = output.detach().cpu()
        return output.to('meta')  # Free GPU memory
    
    def backward_hook(self, module, grad_input, grad_output):
        # Retrieve activations for gradient computation
        act = self.cpu_storage.pop(id(module)).cuda()
        # ... recompute gradients using retrieved activations
```

In practice, this is implemented in DeepSpeed's activation partitioning and in specialized frameworks like Colossal-AI.

## 14. FlashAttention: IO-Aware Exact Attention

### The Standard Attention Memory Problem

Standard self-attention materializes the $N \times N$ attention matrix:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Memory complexity:
$$\text{Standard Attention memory} = O(N^2) \quad \text{(stores } N \times N \text{ attention matrix)}$$

For $N = 128{,}000$ tokens: $128000^2 = 16 \times 10^9$ elements = **32 GB** just for the attention matrix (FP16).

### FlashAttention Algorithm (Dao et al., 2022)

FlashAttention avoids materializing the full attention matrix by tiling the computation to fit in SRAM (fast on-chip memory):

$$\boxed{\text{FlashAttention memory} = O(N) \quad \text{(only stores Q, K, V, O blocks and softmax statistics)}}$$

**Key insight:** Use the online softmax trick compute the softmax incrementally across tiles using the log-sum-exp recurrence:

$$m_i^{\text{new}} = \max(m_i^{\text{old}}, \max_j(S_{ij}))$$

$$\ell_i^{\text{new}} = e^{m_i^{\text{old}} - m_i^{\text{new}}} \ell_i^{\text{old}} + \sum_j e^{S_{ij} - m_i^{\text{new}}}$$

This allows computing the softmax without reading back the full row enabling a single pass over tiles.

### FlashAttention vs Standard Attention

| Metric | Standard Attention | FlashAttention v2 |
|---|---|---|
| HBM reads | $O(N^2)$ | $O(N^2 / M)$ where $M$ = SRAM size |
| HBM writes | $O(N^2)$ | $O(N)$ |
| Memory | $O(N^2)$ | $O(N)$ |
| Speed (A100) | Baseline | 2-4x faster |
| Exactness | Exact | Exact (not approximate!) |

### FlashAttention v2 and v3 Improvements

- **v2:** Better work partitioning across warps, supports causal masking natively. 2x speedup over v1.
- **v3:** H100-specific optimizations, overlaps GEMM and softmax using warpgroup specialization. Up to 3.5x faster than standard attention.

### Gradient Checkpointing for Free

FlashAttention stores only the softmax statistics $(m, \ell)$ and the output $O$, not the full attention matrix. This means it is effectively doing **activation recomputation for attention** at no extra compute cost (Q, K, V are reloaded from HBM, which is fast relative to SRAM storage).

In [7]:
# ============================================================
# FlashAttention: Comparing standard vs efficient attention
# ============================================================
import torch
import torch.nn as nn
import math


def standard_attention(q, k, v, mask=None):
    """
    Standard scaled dot-product attention.
    Memory: O(N^2) materializes full N x N attention matrix.
    """
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, N, N)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = torch.softmax(scores, dim=-1)  # Full N x N matrix in memory
    return torch.matmul(attn_weights, v)


def pytorch_sdpa(q, k, v, is_causal=False):
    """
    PyTorch 2.0+ scaled_dot_product_attention.
    Automatically uses FlashAttention kernel when available on CUDA.
    On CPU: falls back to math implementation.
    """
    return torch.nn.functional.scaled_dot_product_attention(
        q, k, v,
        attn_mask=None,
        dropout_p=0.0,
        is_causal=is_causal,
        # scale=None  # PyTorch 2.1+: custom scale
    )


# Memory complexity analysis
def attention_memory_bytes(seq_len, d_head, n_heads, batch, dtype_bytes=2):
    """Calculate peak memory for standard attention in bytes."""
    # QKV tensors
    qkv_mem = 3 * batch * n_heads * seq_len * d_head * dtype_bytes
    # Attention matrix (the O(N^2) term)
    attn_matrix_mem = batch * n_heads * seq_len * seq_len * dtype_bytes
    # Output
    out_mem = batch * n_heads * seq_len * d_head * dtype_bytes
    return qkv_mem, attn_matrix_mem, out_mem


print("=" * 60)
print("Attention Memory Analysis (BF16, batch=1, heads=32, d_head=128)")
print("=" * 60)
print(f"{'Seq Len':>10} | {'QKV (MB)':>10} | {'Attn Matrix (MB)':>18} | {'Ratio':>8}")
print("-" * 60)
for seq_len in [512, 2048, 8192, 32768, 131072]:
    qkv, attn, out = attention_memory_bytes(seq_len, 128, 32, 1)
    print(f"{seq_len:>10,} | {qkv/1e6:>10.1f} | {attn/1e6:>18.1f} | {attn/qkv:>7.1f}x")

print()
print("FlashAttention stores only O(N) no N x N matrix!")
print()

# Verify numerically that SDPA matches standard attention
B, H, N, D = 2, 4, 64, 32
q = torch.randn(B, H, N, D)
k = torch.randn(B, H, N, D)
v = torch.randn(B, H, N, D)

out_std = standard_attention(q, k, v)
out_sdpa = pytorch_sdpa(q, k, v)

max_diff = (out_std - out_sdpa).abs().max().item()
print(f"Numerical difference (standard vs SDPA): {max_diff:.2e}")
assert max_diff < 1e-5, "Outputs should match!"
print("Outputs match SDPA is numerically equivalent to standard attention")

# PyTorch 2.0 backend selection
print("\nSDPA backends available:")
try:
    backends = torch.backends.cuda.flash_sdp_enabled() if torch.cuda.is_available() else False
    print(f"  FlashAttention:   {torch.cuda.is_available() and backends}")
    print(f"  Memory-efficient: {torch.cuda.is_available()}")
    print(f"  Math fallback:    True (always available)")
except Exception:
    print("  (Backend query requires CUDA)")

Attention Memory Analysis (BF16, batch=1, heads=32, d_head=128)
   Seq Len |   QKV (MB) |   Attn Matrix (MB) |    Ratio
------------------------------------------------------------
       512 |       12.6 |               16.8 |     1.3x
     2,048 |       50.3 |              268.4 |     5.3x
     8,192 |      201.3 |             4295.0 |    21.3x
    32,768 |      805.3 |            68719.5 |    85.3x
   131,072 |     3221.2 |          1099511.6 |   341.3x

FlashAttention stores only O(N) no N x N matrix!

Numerical difference (standard vs SDPA): 4.77e-07
Outputs match SDPA is numerically equivalent to standard attention

SDPA backends available:
  FlashAttention:   False
  Memory-efficient: False
  Math fallback:    True (always available)


## 15. Megatron-LM and Horovod in Practice

### Megatron-LM Architecture

Megatron-LM (NVIDIA) is the framework used to train Megatron-Turing NLG (530B), and forms the backbone of many LLM training systems.

**Key components:**
1. **Tensor parallelism**: Column/row parallel linear layers as described in Section 5
2. **Pipeline parallelism**: 1F1B schedule with interleaved virtual stages
3. **Sequence parallelism**: Splits non-TP operations along sequence dimension
4. **Efficient data loading**: Multimodal dataset mixing with weighted sampling
5. **Custom CUDA kernels**: Fused LayerNorm, fused softmax, rotary embeddings

**Typical 3D parallelism configuration for LLM training:**

```bash
torchrun --nproc_per_node=8 pretrain_gpt.py \
    --tensor-model-parallel-size 4 \
    --pipeline-model-parallel-size 2 \
    --num-layers 32 \
    --hidden-size 4096 \
    --num-attention-heads 32 \
    --seq-length 4096 \
    --micro-batch-size 2 \
    --global-batch-size 1024 \
    --use-flash-attn \
    --bf16
```

### Communication Groups in 3D Parallelism

With TP=4, PP=2, DP=2 on 16 GPUs, the communication groups are:

```
GPU layout (rank):
  Node 0: [0,  1,  2,  3]  [4,  5,  6,  7]   <- PP stage 0 | PP stage 1
  Node 1: [8,  9, 10, 11]  [12, 13, 14, 15]  <- PP stage 0 | PP stage 1

TP groups  (same stage, same node):  {0,1,2,3}, {4,5,6,7}, {8,9,10,11}, ...
PP groups  (same TP rank, different stage): {0,4}, {1,5}, {2,6}, ...
DP groups  (same TP+PP position, different replica): {0,8}, {1,9}, ...
```

### Horovod Ring All-Reduce

Horovod implements Baidu's ring all-reduce:
1. Arrange all $N$ workers in a ring.
2. **Scatter-reduce phase** ($N-1$ steps): Each worker sends 1/N of its gradient to the next, accumulating along the way. After this phase, each worker has the reduced value for one chunk.
3. **All-gather phase** ($N-1$ steps): Broadcast the reduced chunks around the ring.

Total data sent per worker: $2 \cdot \frac{N-1}{N} \cdot \Psi \approx 2\Psi$ same as NCCL ring-allreduce.

In [8]:
# ============================================================
# torch.distributed Communication Primitives
# ============================================================
import torch
import torch.distributed as dist

# These would run inside a distributed process.
# Here we show the API and explain each operation.

comm_ops = {
    "broadcast": {
        "desc": "Send tensor from one rank to all ranks",
        "api": "dist.broadcast(tensor, src=0)",
        "use_case": "Broadcast initial model weights from rank 0",
        "comm_vol": "(N-1) * size",
    },
    "all_reduce": {
        "desc": "Reduce tensor across all ranks, result available on all",
        "api": "dist.all_reduce(tensor, op=dist.ReduceOp.SUM)",
        "use_case": "DDP gradient averaging",
        "comm_vol": "2 * (N-1)/N * size",
    },
    "reduce_scatter": {
        "desc": "Reduce across all ranks, scatter result (each rank gets 1/N)",
        "api": "dist.reduce_scatter(output, input_list)",
        "use_case": "ZeRO-3 gradient sharding, FSDP",
        "comm_vol": "(N-1)/N * size",
    },
    "all_gather": {
        "desc": "Each rank gathers its chunk; all ranks see full tensor",
        "api": "dist.all_gather(tensor_list, tensor)",
        "use_case": "ZeRO-3 parameter reconstruction, FSDP",
        "comm_vol": "(N-1)/N * size",
    },
    "send_recv": {
        "desc": "Point-to-point send/receive between two specific ranks",
        "api": "dist.send(tensor, dst=next_rank) / dist.recv(tensor, src=prev_rank)",
        "use_case": "Pipeline parallelism inter-stage activation passing",
        "comm_vol": "activation_size",
    },
    "isend_irecv": {
        "desc": "Non-blocking point-to-point (overlaps with compute)",
        "api": "req = dist.isend(tensor, dst=...) / dist.irecv(tensor, src=...)",
        "use_case": "PipeDream 1F1B overlap",
        "comm_vol": "activation_size",
    },
}

print("PyTorch Distributed Communication Primitives")
print("=" * 65)
for op, info in comm_ops.items():
    print(f"\n[{op.upper()}]")
    print(f"  Description:  {info['desc']}")
    print(f"  API:          {info['api']}")
    print(f"  Used for:     {info['use_case']}")
    print(f"  Comm volume:  {info['comm_vol']}")


# Simulate all_reduce behavior (single process, manual averaging)
print("\n" + "=" * 65)
print("\nSimulating gradient all-reduce across 4 hypothetical workers:")
workers = [torch.randn(3) for _ in range(4)]
print("Gradients per worker:", [w.tolist() for w in workers])

# All-reduce = sum then divide (mean)
reduced = torch.stack(workers).mean(dim=0)
print(f"After all-reduce (mean): {reduced.tolist()}")
print("Each worker uses this identical gradient for its optimizer step.")

# Simulate reduce_scatter behavior
print("\nSimulating reduce_scatter (ZeRO-3 gradient sharding):")
summed = torch.stack(workers).sum(dim=0)  # Sum gradients
chunks = summed.chunk(len(workers))
for i, chunk in enumerate(chunks):
    print(f"  Worker {i} owns gradient shard: {chunk.tolist()}")

PyTorch Distributed Communication Primitives

[BROADCAST]
  Description:  Send tensor from one rank to all ranks
  API:          dist.broadcast(tensor, src=0)
  Used for:     Broadcast initial model weights from rank 0
  Comm volume:  (N-1) * size

[ALL_REDUCE]
  Description:  Reduce tensor across all ranks, result available on all
  API:          dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
  Used for:     DDP gradient averaging
  Comm volume:  2 * (N-1)/N * size

[REDUCE_SCATTER]
  Description:  Reduce across all ranks, scatter result (each rank gets 1/N)
  API:          dist.reduce_scatter(output, input_list)
  Used for:     ZeRO-3 gradient sharding, FSDP
  Comm volume:  (N-1)/N * size

[ALL_GATHER]
  Description:  Each rank gathers its chunk; all ranks see full tensor
  API:          dist.all_gather(tensor_list, tensor)
  Used for:     ZeRO-3 parameter reconstruction, FSDP
  Comm volume:  (N-1)/N * size

[SEND_RECV]
  Description:  Point-to-point send/receive between two specific 

## 16. Putting It All Together: Choosing the Right Strategy

### Decision Framework

```
Does the model fit on one GPU?
├── YES → Use DDP (data parallelism only)
│         + AMP (BF16/FP16)
│         + Gradient checkpointing if memory-tight
└── NO  → How large is the model?
          ├── 1B-10B params → FSDP FULL_SHARD (ZeRO-3)
          │                   + BF16 + gradient checkpointing
          ├── 10B-100B params → FSDP or DeepSpeed ZeRO-3
          │                    + TP within node (Megatron or custom)
          └── 100B+ params → 3D parallelism (TP + PP + DP)
                            Megatron-LM or DeepSpeed + Megatron
```

### Memory Budget Planning

For a model with $\Psi$ parameters across $N$ GPUs with ZeRO-3:

| Component | Memory formula |
|---|---|
| Model states (ZeRO-3) | $\frac{16\Psi}{N}$ bytes |
| Activations (w/ gradient ckpt) | $\approx \frac{L}{p} \cdot \text{seq\_len} \cdot d_{\text{model}} \cdot \text{batch} \cdot 2$ bytes |
| Temporary buffers | ~1-3 GB |

### Real-World Example: LLaMA-3 8B Fine-tuning

$\Psi = 8 \times 10^9$, target: fit on 8x A100 40GB

**With FSDP FULL_SHARD + BF16 + gradient checkpointing:**
- Model states: $\frac{16 \times 8 \times 10^9}{8} = 16$ GB per GPU
- Activations (seq=4096, batch=2, L=32): $\approx 4$ GB
- Buffers: $\approx 2$ GB
- **Total: ~22 GB** fits on 40GB A100!

### Communication Bandwidth Requirements

| Strategy | Communication | Typical bottleneck |
|---|---|---|
| DDP | All-reduce $2\Psi$ per step | Inter-node bandwidth |
| FSDP ZeRO-3 | All-gather + reduce-scatter $3\Psi$ | Intra-node NVLink |
| TP (Megatron) | 2 all-reduces per transformer layer | NVLink (must be intra-node) |
| PP | Micro-batch activations per stage | Point-to-point, inter-node OK |

In [9]:
# ============================================================
# Distributed Training Memory Calculator
# ============================================================
import math


def calc_model_states_memory(
    params: float,        # Number of parameters (e.g., 7e9 for 7B)
    n_gpus: int,          # Total number of GPUs
    zero_stage: int = 3,  # ZeRO stage: 0 (DDP), 1, 2, or 3
    mixed_precision: bool = True,  # BF16/FP16 + FP32 optimizer
) -> float:
    """Estimate model state memory per GPU in GB."""
    if mixed_precision:
        # 2 (params) + 2 (grads) + 4 (master) + 4 (m) + 4 (v) = 16 bytes/param
        bytes_per_param_full = 16
    else:
        # 4 (params) + 4 (grads) + 4 (m) + 4 (v) = 16 bytes/param
        bytes_per_param_full = 16

    if zero_stage == 0:  # Pure DDP
        mem = bytes_per_param_full * params
    elif zero_stage == 1:  # Shard optimizer states only
        opt_states = 12 * params  # FP32 master + m + v
        rest = 4 * params         # FP16 params + FP16 grads
        mem = rest + opt_states / n_gpus
    elif zero_stage == 2:  # Shard optimizer states + gradients
        opt_states = 12 * params
        grads = 2 * params
        params_mem = 2 * params
        mem = params_mem + (grads + opt_states) / n_gpus
    elif zero_stage == 3:  # Shard everything
        mem = bytes_per_param_full * params / n_gpus
    else:
        raise ValueError(f"Unknown ZeRO stage: {zero_stage}")

    return mem / 1e9  # Convert to GB


def calc_activation_memory(
    seq_len: int,
    batch_size: int,
    d_model: int,
    n_layers: int,
    n_heads: int,
    gradient_checkpointing: bool = True,
) -> float:
    """Rough estimate of activation memory in GB (BF16)."""
    # Per-layer activation (rough): 34 * seq_len * d_model bytes (BF16)
    # With checkpointing: only 1 layer's activations live at once
    per_layer_bytes = 34 * seq_len * d_model * batch_size * 2  # BF16
    if gradient_checkpointing:
        total = per_layer_bytes  # Only one layer at a time
    else:
        total = per_layer_bytes * n_layers
    return total / 1e9


# === Model Comparison Table ===
models = [
    ("LLaMA-3 1B",  1e9,   16, 2048, 32, 4),
    ("LLaMA-3 8B",  8e9,   32, 4096, 32, 8),
    ("LLaMA-3 70B", 70e9,  80, 8192, 64, 16),
    ("GPT-3 175B",  175e9, 96, 12288, 96, 64),
]

print(f"{'Model':>15} | {'Stage':>5} | {'GPU':>4} | {'Model States (GB)':>18} | {'Activations (GB)':>17} | {'Total (GB)':>11}")
print("-" * 85)

for name, params, n_layers, d_model, n_heads, n_gpus in models:
    for stage in [0, 3]:
        ms = calc_model_states_memory(params, n_gpus, zero_stage=stage)
        act = calc_activation_memory(4096, 1, d_model, n_layers, n_heads, gradient_checkpointing=True)
        total = ms + act + 2.0  # +2 GB for buffers
        stage_str = "DDP" if stage == 0 else "ZeRO-3"
        print(f"{name:>15} | {stage_str:>5} | {n_gpus:>4} | {ms:>18.1f} | {act:>17.2f} | {total:>11.1f}")
    print()

print("Note: Activations estimated with gradient checkpointing enabled.")
print("Add ~3-5x activations if gradient checkpointing is disabled.")

          Model | Stage |  GPU |  Model States (GB) |  Activations (GB) |  Total (GB)
-------------------------------------------------------------------------------------
     LLaMA-3 1B |   DDP |    4 |               16.0 |              0.57 |        18.6
     LLaMA-3 1B | ZeRO-3 |    4 |                4.0 |              0.57 |         6.6

     LLaMA-3 8B |   DDP |    8 |              128.0 |              1.14 |       131.1
     LLaMA-3 8B | ZeRO-3 |    8 |               16.0 |              1.14 |        19.1

    LLaMA-3 70B |   DDP |   16 |             1120.0 |              2.28 |      1124.3
    LLaMA-3 70B | ZeRO-3 |   16 |               70.0 |              2.28 |        74.3

     GPT-3 175B |   DDP |   64 |             2800.0 |              3.42 |      2805.4
     GPT-3 175B | ZeRO-3 |   64 |               43.8 |              3.42 |        49.2

Note: Activations estimated with gradient checkpointing enabled.
Add ~3-5x activations if gradient checkpointing is disabled.


## 17. Practical Tips and Common Pitfalls

### Effective Batch Size and Learning Rate Scaling

When scaling from 1 GPU to $N$ GPUs with DDP, the global batch size increases by $N$. Common scaling rules:

**Linear scaling rule (Goyal et al.):**
$$\text{lr}_{\text{new}} = \text{lr}_{\text{base}} \times \frac{\text{batch\_size}_{\text{new}}}{\text{batch\_size}_{\text{base}}}$$

**Square root scaling (often better for large batches):**
$$\text{lr}_{\text{new}} = \text{lr}_{\text{base}} \times \sqrt{\frac{\text{batch\_size}_{\text{new}}}{\text{batch\_size}_{\text{base}}}}$$

**Warmup:** Always use learning rate warmup (500-2000 steps) when training at large batch size the model is sensitive to large gradient steps early.

### Common Bugs

| Bug | Symptom | Fix |
|---|---|---|
| Forgetting `sampler.set_epoch(epoch)` | Model trains on same data order each epoch | Always call `set_epoch` |
| Only calling `optimizer.step()` on rank 0 | Models diverge across ranks | Call on all ranks |
| Saving checkpoints on all ranks | Race condition / corrupted file | Save only on rank 0 |
| Missing `dist.barrier()` before loading ckpt | Some ranks skip loading | Add barrier |
| `find_unused_parameters=True` always on | Massive slowdown (~2x) | Only use when needed |
| FP16 NaN loss | Gradient overflow | Use BF16 or check loss scale |
| NCCL timeout | Slow data loading stalls one rank | Increase timeout, fix data pipeline |

### Profiling Distributed Training

```python
# PyTorch Profiler with distributed support
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=3),
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profile'),
    record_shapes=True,
    with_stack=True,
) as prof:
    for step, (x, y) in enumerate(loader):
        train_step(model, x, y, optimizer)
        prof.step()
```

Use **NVIDIA Nsight Systems** for GPU-level profiling: `nsys profile -t cuda,nvtx python train.py`

### MFU (Model FLOP Utilization)

The key metric for distributed training efficiency:

$$\text{MFU} = \frac{\text{achieved FLOP/s}}{\text{peak FLOP/s}} = \frac{6\Psi \cdot B / T}{N_{\text{GPU}} \cdot \text{peak FLOP/s}}$$

where $B$ = tokens per batch, $T$ = time per step. Target: >40% MFU. Production LLM training: 35-55% MFU.

In [10]:
# ============================================================
# MFU Calculator and Training Throughput Estimator
# ============================================================
import math


def estimate_flops_per_token(n_layers, d_model, n_heads, d_ff=None, seq_len=2048):
    """
    Estimate FLOPs per token for a GPT-style transformer.
    Based on: 6 * params (for matmuls, roughly).
    More detailed estimate:
    """
    if d_ff is None:
        d_ff = 4 * d_model
    d_head = d_model // n_heads

    # Per-layer FLOPs (forward only, multiply by 3 for fwd+bwd)
    # Attention: QKV projections + attention scores + output proj
    attn_flops = 2 * d_model * 3 * d_model  # QKV
    attn_flops += 2 * seq_len * d_model      # Attention scores (per token in seq)
    attn_flops += 2 * d_model * d_model      # Output projection

    # FFN: two linear layers
    ffn_flops = 2 * 2 * d_model * d_ff

    per_layer = attn_flops + ffn_flops
    total_fwd = n_layers * per_layer
    return total_fwd * 3  # forward + backward is ~3x forward


def compute_mfu(
    params: float,         # Model parameters
    tokens_per_batch: int, # Global batch size in tokens
    step_time_s: float,    # Time per step in seconds
    n_gpus: int,
    gpu_flops: float,      # Peak FP16/BF16 FLOP/s per GPU
) -> float:
    """Compute Model FLOP Utilization."""
    # FLOPs per step (6 * params * tokens is the standard estimate)
    flops_per_step = 6 * params * tokens_per_batch
    achieved_flops = flops_per_step / step_time_s
    peak_flops = n_gpus * gpu_flops
    return achieved_flops / peak_flops


# GPU specs (peak BF16 FLOP/s)
gpu_specs = {
    "A100 80GB": 312e12,    # 312 TFLOP/s BF16
    "H100 SXM": 989e12,     # 989 TFLOP/s BF16
    "V100 32GB": 125e12,    # 125 TFLOP/s FP16
    "RTX 4090": 82.6e12,    # 82.6 TFLOP/s BF16
    "RTX 3090": 35.6e12,    # 35.6 TFLOP/s FP16
}

print("GPU Peak Performance:")
for gpu, flops in gpu_specs.items():
    print(f"  {gpu:<15}: {flops/1e12:.1f} TFLOP/s (BF16/FP16)")

print()
print("MFU Analysis for LLaMA-3 8B Training")
print("-" * 55)
params = 8e9
batch_tokens = 4096 * 4  # seq_len=4096, batch=4

scenarios = [
    ("8x A100, step=2s, good", 8, 312e12, 2.0),
    ("8x A100, step=4s, bottleneck", 8, 312e12, 4.0),
    ("1x A100, step=18s, single GPU", 1, 312e12, 18.0),
    ("8x H100, step=0.8s, H100", 8, 989e12, 0.8),
]

for label, n_gpus, gpu_flops, step_time in scenarios:
    mfu = compute_mfu(params, batch_tokens, step_time, n_gpus, gpu_flops)
    tokens_per_sec = batch_tokens / step_time
    print(f"  {label}")
    print(f"    MFU: {mfu*100:.1f}%  |  {tokens_per_sec:.0f} tokens/s")

print()
print("Target MFU > 40% indicates efficient distributed training.")
print("Below 30% suggests communication, I/O, or load balancing issues.")

GPU Peak Performance:
  A100 80GB      : 312.0 TFLOP/s (BF16/FP16)
  H100 SXM       : 989.0 TFLOP/s (BF16/FP16)
  V100 32GB      : 125.0 TFLOP/s (BF16/FP16)
  RTX 4090       : 82.6 TFLOP/s (BF16/FP16)
  RTX 3090       : 35.6 TFLOP/s (BF16/FP16)

MFU Analysis for LLaMA-3 8B Training
-------------------------------------------------------
  8x A100, step=2s, good
    MFU: 15.8%  |  8192 tokens/s
  8x A100, step=4s, bottleneck
    MFU: 7.9%  |  4096 tokens/s
  1x A100, step=18s, single GPU
    MFU: 14.0%  |  910 tokens/s
  8x H100, step=0.8s, H100
    MFU: 12.4%  |  20480 tokens/s

Target MFU > 40% indicates efficient distributed training.
Below 30% suggests communication, I/O, or load balancing issues.


## 18. Summary

### Key Takeaways

**Parallelism strategies** address both memory and compute limitations:

| Strategy | Memory saving | Compute scaling | Key cost |
|---|---|---|---|
| DDP | None | Linear with $N$ GPUs | All-reduce comm |
| ZeRO-3 / FSDP | $N \times$ reduction | None | 1.5x DDP comm |
| Tensor Parallelism | $N$ within group | Linear | 2 all-reduce / layer |
| Pipeline Parallelism | $N$ stages | Near-linear | Bubble overhead |

**Memory optimization hierarchy** (apply in order):
1. Mixed precision (BF16) **free** 2x memory saving
2. Gradient checkpointing ~33% compute overhead, large memory saving
3. FSDP / ZeRO-3 linear memory scaling with GPU count
4. CPU offload large saving, PCIe bandwidth cost
5. Activation offloading for extreme memory pressure

**Algorithm innovations:**
- **FlashAttention**: $O(N^2) \rightarrow O(N)$ memory, 2-4x faster use it always
- **Gradient checkpointing**: Free memory at 33% compute cost standard for large models
- **BF16**: No overflow risk, free 2x memory preferred over FP16 on Ampere+

### The LLM Training Stack (2024)

```
Application:    Megatron-LM / NeMo / Llama-Factory
Parallelism:    3D (TP + PP + DP/ZeRO)
Memory:         ZeRO-3 / FSDP + gradient checkpointing
Precision:      BF16 (weights/grads) + FP32 (master weights/optimizer)
Attention:      FlashAttention v2/v3
Communication:  NCCL (intra-node NVLink, inter-node InfiniBand)
Hardware:       H100/A100 NVLink clusters
```

## Additional Learning Resources

### Foundational Papers

1. **ZeRO: Memory Optimizations Toward Training Trillion Parameter Models**  
   Rajbhandari et al., Microsoft, 2020.  
   https://arxiv.org/abs/1910.02054

2. **Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism**  
   Shoeybi et al., NVIDIA, 2019.  
   https://arxiv.org/abs/1909.08053

3. **Efficient Large Scale Language Modeling with Mixtures of Experts (Reducing Pipeline Bubbles)**  
   Artetxe et al., 2021 covers interleaved pipeline schedule.  
   https://arxiv.org/abs/2112.10684

4. **GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism**  
   Huang et al., Google Brain, 2019.  
   https://arxiv.org/abs/1811.06965

5. **PipeDream: Generalized Pipeline Parallelism for DNN Training**  
   Narayanan et al., Microsoft Research, 2019.  
   https://arxiv.org/abs/1806.03377

6. **FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness**  
   Dao et al., 2022.  
   https://arxiv.org/abs/2205.14135

7. **FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning**  
   Dao, 2023.  
   https://arxiv.org/abs/2307.08691

8. **Reducing Activation Recomputation in Large Transformer Models (Sequence Parallelism)**  
   Korthikanti et al., NVIDIA, 2022.  
   https://arxiv.org/abs/2205.05198

9. **Horovod: fast and easy distributed deep learning in TensorFlow**  
   Sergeev & Del Balso, Uber, 2018.  
   https://arxiv.org/abs/1802.05799

10. **ZeRO-Infinity: Breaking the GPU Memory Wall for Extreme Scale Deep Learning**  
    Rajbhandari et al., Microsoft, 2021.  
    https://arxiv.org/abs/2104.07857

### Official Documentation

- **PyTorch Distributed**: https://pytorch.org/docs/stable/distributed.html
- **PyTorch FSDP Tutorial**: https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html
- **DeepSpeed Documentation**: https://www.deepspeed.ai/docs/config-json/
- **DeepSpeed ZeRO Tutorial**: https://www.deepspeed.ai/tutorials/zero/
- **Megatron-LM GitHub**: https://github.com/NVIDIA/Megatron-LM
- **FlashAttention GitHub**: https://github.com/Dao-AILab/flash-attention
- **NCCL Documentation**: https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/index.html

### Tutorials and Courses

- **Hugging Face: Efficient Training on Multiple GPUs**  
  https://huggingface.co/docs/transformers/perf_train_gpu_many

- **Andrej Karpathy: Let's reproduce GPT-2 (distributed training section)**  
  https://www.youtube.com/watch?v=l8pRSuU81PU

- **Lilian Weng: Large-scale Transformer Model Training**  
  https://lilianweng.github.io/posts/2021-09-25-train-large/

- **EleutherAI: Cookbook for Large Language Model Training**  
  https://blog.eleuther.ai/

- **Tim Dettmers: How to Train Really Large Models on Many GPUs**  
  https://huggingface.co/blog/large-language-models

### Libraries to Explore

| Library | Purpose |
|---|---|
| `torch.distributed` | Core PyTorch distributed training |
| `torch.distributed.fsdp` | FSDP (ZeRO-3 in PyTorch) |
| `deepspeed` | ZeRO + CPU/NVMe offload |
| `megatron-lm` | 3D parallelism for LLMs |
| `flash-attn` | FlashAttention CUDA kernels |
| `accelerate` (HuggingFace) | High-level distributed training API |
| `colossalai` | Unified parallelism framework |
| `fairscale` | Facebook Research distributed primitives |
| `torchrun` | PyTorch distributed launcher |
| `horovod` | Uber's ring-allreduce framework |